# 04 - Taxonomy Drift Detection Demo

Companion notebook to `07-model-and-taxonomy-staleness.md`. Implements the drift-detection design from
chapter 07: a synthetic comparison between a "training-time" taxonomy category distribution/accuracy and
a "current" one, flagging categories that have drifted beyond a threshold -- the scheduled job chapter 07
proposes to catch taxonomy drift before a compliance reviewer catches it manually.

All data here is synthetic, constructed to demonstrate the detection logic clearly. No real documents,
no real model, no network calls.

## 1. Synthetic training-time taxonomy state

A registry entry (chapter 06/07) for a deployed adapter, tagged with the taxonomy version it was trained
against, its per-category classification distribution at training time (what fraction of the training
set fell into each category), and its per-category accuracy on the domain evaluation set (chapter 05) at
that time.

In [1]:
registry_entry = {
    "artifact_id": "mixtral-8x7b-regtaxonomy-adapter-v14",
    "taxonomy_version": "reg-taxonomy-v7.2",
    "trained_at": "2026-05-02T00:00:00Z",
}

# Training-time state: category -> (fraction of training/eval examples, per-category F1)
training_time_state = {
    "AML-CDD-CORR-BANKING": (0.18, 0.94),
    "SANCTIONS-SCREENING": (0.15, 0.92),
    "CAPITAL-MARKETS-CONDUCT": (0.22, 0.90),
    "DATA-PRIVACY-CROSS-BORDER": (0.12, 0.88),
    "CONSUMER-DUTY-DISCLOSURE": (0.20, 0.91),
    "OPERATIONAL-RESILIENCE": (0.13, 0.89),
}

print("Registry entry: {} (taxonomy_version={})".format(registry_entry["artifact_id"], registry_entry["taxonomy_version"]))
print()
print("{:28s} {:>18s} {:>14s}".format("Category", "Train-time share", "Train-time F1"))
print("-" * 64)
for category, (share, f1) in training_time_state.items():
    print("{:28s} {:17.1%} {:14.1%}".format(category, share, f1))


Registry entry: mixtral-8x7b-regtaxonomy-adapter-v14 (taxonomy_version=reg-taxonomy-v7.2)

Category                       Train-time share  Train-time F1
----------------------------------------------------------------
AML-CDD-CORR-BANKING                     18.0%          94.0%
SANCTIONS-SCREENING                      15.0%          92.0%
CAPITAL-MARKETS-CONDUCT                  22.0%          90.0%
DATA-PRIVACY-CROSS-BORDER                12.0%          88.0%
CONSUMER-DUTY-DISCLOSURE                 20.0%          91.0%
OPERATIONAL-RESILIENCE                   13.0%          89.0%


## 2. Synthetic "current" state, months later

Two things have happened since training: the taxonomy gained a new category the deployed adapter has
never seen (`OPERATIONAL-RESILIENCE-THIRD-PARTY`, a split-off from `OPERATIONAL-RESILIENCE`), and
accuracy has quietly degraded on a couple of categories -- one mildly (normal model drift), one sharply
(the kind of drop chapter 07 says a human reviewer would otherwise have to catch manually).

In [2]:
live_taxonomy_version = "reg-taxonomy-v7.5"  # the taxonomy has moved on since v7.2

# Current production state: category -> (current live share of incoming documents, current measured F1
# against a held-out validation set kept current with the live taxonomy -- chapter 07, Part 4)
current_state = {
    "AML-CDD-CORR-BANKING": (0.17, 0.93),               # essentially stable
    "SANCTIONS-SCREENING": (0.14, 0.90),                 # mild, expected drift
    "CAPITAL-MARKETS-CONDUCT": (0.21, 0.71),             # sharp accuracy drop -- the real problem
    "DATA-PRIVACY-CROSS-BORDER": (0.11, 0.87),           # mild, expected drift
    "CONSUMER-DUTY-DISCLOSURE": (0.09, 0.90),            # share dropped sharply -- volume shift
    "OPERATIONAL-RESILIENCE": (0.07, 0.86),              # share dropped -- some volume moved to the new category
    "OPERATIONAL-RESILIENCE-THIRD-PARTY": (0.06, None),  # BRAND NEW category -- adapter was never trained on it, no F1 to compare
}

print("Live taxonomy_version: {}  (deployed adapter trained against: {})".format(
    live_taxonomy_version, registry_entry["taxonomy_version"]))
print()
if live_taxonomy_version != registry_entry["taxonomy_version"]:
    print("TAXONOMY VERSION MISMATCH detected: the live taxonomy has advanced past this adapter's")
    print("training-time taxonomy_version -- chapter 07's standing, explicitly checkable signal.")


Live taxonomy_version: reg-taxonomy-v7.5  (deployed adapter trained against: reg-taxonomy-v7.2)

TAXONOMY VERSION MISMATCH detected: the live taxonomy has advanced past this adapter's
training-time taxonomy_version -- chapter 07's standing, explicitly checkable signal.


## 3. The drift-detection job: distribution shift and accuracy drop, per category

Implements chapter 07's proposed scheduled job: compare live classification-share distribution against
the training-time distribution (flagging unexpected volume shifts), and compare current measured
accuracy against training-time accuracy (flagging material accuracy drops) -- plus a hard flag for any
category present now that the adapter was never trained on at all.

In [3]:
SHARE_DRIFT_THRESHOLD = 0.03    # flag if a category's share of documents shifts by more than this (absolute)
ACCURACY_DROP_THRESHOLD = 0.05  # flag if per-category accuracy drops by more than this (absolute)

def detect_drift(training_time_state, current_state, share_threshold, accuracy_threshold):
    flags = []
    all_categories = set(training_time_state) | set(current_state)
    for category in sorted(all_categories):
        train_entry = training_time_state.get(category)
        current_entry = current_state.get(category)

        if train_entry is None:
            flags.append((category, "NEW_CATEGORY_NOT_IN_TRAINING", None, None))
            continue

        train_share, train_f1 = train_entry
        current_share, current_f1 = current_entry if current_entry else (0.0, None)

        share_delta = current_share - train_share
        if abs(share_delta) > share_threshold:
            flags.append((category, "SHARE_DRIFT", share_delta, None))

        if current_f1 is None:
            flags.append((category, "NO_CURRENT_ACCURACY_DATA", None, None))
        else:
            accuracy_delta = current_f1 - train_f1
            if accuracy_delta < -accuracy_threshold:
                flags.append((category, "ACCURACY_DROP", None, accuracy_delta))

    return flags


flags = detect_drift(training_time_state, current_state, SHARE_DRIFT_THRESHOLD, ACCURACY_DROP_THRESHOLD)

print("Drift-detection job results ({} flag(s) raised):".format(len(flags)))
print("-" * 70)
for category, flag_type, share_delta, accuracy_delta in flags:
    if flag_type == "NEW_CATEGORY_NOT_IN_TRAINING":
        print("[{}] {}: this category did not exist at training time -- adapter cannot classify it correctly.".format(flag_type, category))
    elif flag_type == "SHARE_DRIFT":
        print("[{}] {}: document share shifted by {:+.1%} vs. training time.".format(flag_type, category, share_delta))
    elif flag_type == "ACCURACY_DROP":
        print("[{}] {}: accuracy dropped by {:.1%} vs. training-time F1.".format(flag_type, category, accuracy_delta))

assert any(f[0] == "OPERATIONAL-RESILIENCE-THIRD-PARTY" and f[1] == "NEW_CATEGORY_NOT_IN_TRAINING" for f in flags)
assert any(f[0] == "CAPITAL-MARKETS-CONDUCT" and f[1] == "ACCURACY_DROP" for f in flags)
print()
print("Confirmed: the job catches both the brand-new taxonomy category the adapter has never seen, and")
print("the sharp, otherwise-invisible accuracy drop on CAPITAL-MARKETS-CONDUCT -- exactly the kind of")
print("silent degradation chapter 07 says would otherwise wait for a compliance reviewer to catch by hand.")


Drift-detection job results (4 flag(s) raised):
----------------------------------------------------------------------
[ACCURACY_DROP] CAPITAL-MARKETS-CONDUCT: accuracy dropped by -19.0% vs. training-time F1.
[SHARE_DRIFT] CONSUMER-DUTY-DISCLOSURE: document share shifted by -11.0% vs. training time.
[SHARE_DRIFT] OPERATIONAL-RESILIENCE: document share shifted by -6.0% vs. training time.
[NEW_CATEGORY_NOT_IN_TRAINING] OPERATIONAL-RESILIENCE-THIRD-PARTY: this category did not exist at training time -- adapter cannot classify it correctly.

Confirmed: the job catches both the brand-new taxonomy category the adapter has never seen, and
the sharp, otherwise-invisible accuracy drop on CAPITAL-MARKETS-CONDUCT -- exactly the kind of
silent degradation chapter 07 says would otherwise wait for a compliance reviewer to catch by hand.


## 4. What this job would trigger, in a real design

Per chapter 07, this job's output feeds a concrete action, not just a dashboard: a `taxonomy_version`
mismatch plus one or more `ACCURACY_DROP`/`NEW_CATEGORY_NOT_IN_TRAINING` flags should open a retraining
work item tagged to the live taxonomy version, and -- if the accuracy drop is severe enough on a
high-volume category -- should be treated as a production quality incident, not just a backlog item.

In [4]:
SEVERE_ACCURACY_DROP_THRESHOLD = 0.10  # a drop this large, on a high-share category, is an incident, not a backlog item

def summarize_action(flags, current_state, severe_threshold):
    incidents = []
    backlog_items = []
    for category, flag_type, share_delta, accuracy_delta in flags:
        if flag_type == "ACCURACY_DROP" and accuracy_delta is not None and abs(accuracy_delta) >= severe_threshold:
            share = current_state.get(category, (0.0, None))[0]
            incidents.append((category, accuracy_delta, share))
        else:
            backlog_items.append((category, flag_type))
    return incidents, backlog_items


incidents, backlog_items = summarize_action(flags, current_state, SEVERE_ACCURACY_DROP_THRESHOLD)

print("Recommended action:")
if incidents:
    print("  PRODUCTION QUALITY INCIDENT(S):")
    for category, accuracy_delta, share in incidents:
        print("    - {}: accuracy dropped {:.1%}, currently {:.1%} of live document volume".format(category, accuracy_delta, share))
else:
    print("  No severe accuracy drops above the incident threshold.")

print("  RETRAINING BACKLOG ITEM(S) (tagged to taxonomy_version={}):".format(live_taxonomy_version))
for category, flag_type in backlog_items:
    print("    - {} ({})".format(category, flag_type))


Recommended action:
  PRODUCTION QUALITY INCIDENT(S):
    - CAPITAL-MARKETS-CONDUCT: accuracy dropped -19.0%, currently 21.0% of live document volume
  RETRAINING BACKLOG ITEM(S) (tagged to taxonomy_version=reg-taxonomy-v7.5):
    - CONSUMER-DUTY-DISCLOSURE (SHARE_DRIFT)
    - OPERATIONAL-RESILIENCE (SHARE_DRIFT)
    - OPERATIONAL-RESILIENCE-THIRD-PARTY (NEW_CATEGORY_NOT_IN_TRAINING)


## Summary

| Section | Demonstrates | Matches |
|---|---|---|
| 1-2 | Registry entry with a training-time taxonomy_version vs. a live taxonomy_version that has moved on | Chapter 07, Part 4's registry design (taxonomy-version tags) |
| 3 | A scheduled job flagging share drift, accuracy drop, and brand-new untrained categories | Chapter 07, Part 4's proposed drift-detection job |
| 4 | Severe drops routed as incidents; smaller ones as a retraining backlog | Chapter 07's design turning detection into a concrete, prioritized action |

The sharpest point this notebook makes concrete: `CAPITAL-MARKETS-CONDUCT`'s accuracy dropped from 90%
to 71% with **no change in its document share** -- nothing about incoming volume would have hinted at
the problem. Only comparing current accuracy against a held-out, taxonomy-version-tagged validation set
(chapter 07, Part 4) catches it -- which is exactly why chapter 07 proposes that comparison as a
scheduled job, not an occasional spot-check.